In [1]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║  NSGA-II Simple  vs  NSGA-II + HEFT Hybride                                ║
║  Workflow : workflow_10 (30 tâches × 5 VMs)                                ║
║  ──────────────────────────────────────────────────────────────────────────║
║  ① NSGA-II SIMPLE                                                          ║
║    • Encodage  : ordre topologique Kahn aléatoire + VM map                 ║
║    • Respect des dépendances sans encodage par niveaux                     ║
║    • Normalisation min-max des objectifs                                   ║
║    • Crowding distance standard (sans poids dynamiques)                    ║
║                                                                            ║
║  ② NSGA-II + HEFT HYBRIDE (Notre méthode)                                  ║
║    • Encodage  : topologique par niveaux (jamais invalide)                 ║
║    • Croisement OX niveau par niveau + VM uniforme                         ║
║    • Mutation  : swap intra-niveau + réassignation VM                      ║
║    • Init      : HEFT exact + 30% HEFT perturbé + 70% aléatoire           ║
║    • LS HEFT   : local search chemin critique                              ║
║    • Normalisation + poids dynamiques auto-équilibrés                      ║
║    • Crowding distance pondérée par poids dynamiques                       ║
║                                                                            ║
║  MÉTRIQUES  : HV normalisé ∈ [0,1.331] · IGD+ · |Pareto|                 ║
║  GRAPHES    : Fronts Pareto 2D (3 projections) + 3D                       ║
║  RUNS       : 20 runs + statistiques + run principal seed=42               ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import random, copy, math, time
from typing import Dict, List, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# ══════════════════════════════════════════════════════════════════════
#  1. DONNÉES
# ══════════════════════════════════════════════════════════════════════

def workflow_10() -> Dict:
    tasks = {

        # ==========================
        # NIVEAU 1 (sources)
        # ==========================
        "T1": {"duration": 12, "deps": [], "data": {}},
        "T2": {"duration": 10, "deps": [], "data": {}},
        "T3": {"duration": 14, "deps": [], "data": {}},
        "T4": {"duration": 11, "deps": [], "data": {}},
        "T5": {"duration": 13, "deps": [], "data": {}},

        # ==========================
        # NIVEAU 2
        # ==========================
        "T6": {"duration": 9, "deps": ["T1"], "data": {"T1": 20}},
        "T7": {"duration": 8, "deps": ["T1"], "data": {"T1": 15}},
        "T8": {"duration": 10, "deps": ["T2"], "data": {"T2": 25}},
        "T9": {"duration": 7, "deps": ["T2"], "data": {"T2": 10}},
        "T10": {"duration": 9, "deps": ["T3"], "data": {"T3": 20}},
        "T11": {"duration": 8, "deps": ["T3"], "data": {"T3": 15}},
        "T12": {"duration": 11, "deps": ["T4"], "data": {"T4": 20}},
        "T13": {"duration": 7, "deps": ["T5"], "data": {"T5": 10}},
        "T14": {"duration": 6, "deps": ["T5"], "data": {"T5": 15}},

        # ==========================
        # NIVEAU 3 (fusion)
        # ==========================
        "T15": {"duration": 12, "deps": ["T6", "T8"], "data": {"T6": 15, "T8": 15}},
        "T16": {"duration": 10, "deps": ["T7", "T9"], "data": {"T7": 10, "T9": 10}},
        "T17": {"duration": 11, "deps": ["T10", "T11"], "data": {"T10": 10, "T11": 10}},
        "T18": {"duration": 13, "deps": ["T12"], "data": {"T12": 20}},
        "T19": {"duration": 9, "deps": ["T13", "T14"], "data": {"T13": 10, "T14": 10}},

        # ==========================
        # NIVEAU 4 (fusion avancée)
        # ==========================
        "T20": {"duration": 14, "deps": ["T15", "T16"], "data": {"T15": 20, "T16": 20}},
        "T21": {"duration": 12, "deps": ["T17"], "data": {"T17": 15}},
        "T22": {"duration": 13, "deps": ["T18"], "data": {"T18": 20}},
        "T23": {"duration": 10, "deps": ["T19"], "data": {"T19": 15}},

        # ==========================
        # NIVEAU 5
        # ==========================
        "T24": {"duration": 11, "deps": ["T20"], "data": {"T20": 20}},
        "T25": {"duration": 9, "deps": ["T20"], "data": {"T20": 10}},
        "T26": {"duration": 12, "deps": ["T21", "T22"], "data": {"T21": 15, "T22": 15}},
        "T27": {"duration": 10, "deps": ["T23"], "data": {"T23": 10}},

        # ==========================
        # NIVEAU FINAL
        # ==========================
        "T28": {"duration": 14, "deps": ["T24", "T25"], "data": {"T24": 10, "T25": 10}},
        "T29": {"duration": 13, "deps": ["T26"], "data": {"T26": 20}},
        "T30": {"duration": 15, "deps": ["T27", "T28", "T29"],
                "data": {"T27": 10, "T28": 15, "T29": 20}},
    }

    return tasks


def vms_5() -> Dict:
    return {
        "VM1": {"speed": 1.0, "cost": 2, "power": 30},
        "VM2": {"speed": 1.2, "cost": 2.5, "power": 35},
        "VM3": {"speed": 1.5, "cost": 3, "power": 40},
        "VM4": {"speed": 1.8, "cost": 4, "power": 50},
        "VM5": {"speed": 2.2, "cost": 6, "power": 70},
    }

# ══════════════════════════════════════════════════════════════════════
#  2. STRUCTURE TOPOLOGIQUE
# ══════════════════════════════════════════════════════════════════════

def compute_levels(tasks: Dict) -> Dict[str, int]:
    in_deg = {t: len(tasks[t]["deps"]) for t in tasks}
    succ   = {t: [] for t in tasks}
    for t in tasks:
        for dep in tasks[t]["deps"]:
            succ[dep].append(t)
    level = {t: 0 for t in tasks}
    queue = [t for t in tasks if in_deg[t] == 0]
    while queue:
        node = queue.pop(0)
        for s in succ[node]:
            level[s] = max(level[s], level[node] + 1)
            in_deg[s] -= 1
            if in_deg[s] == 0:
                queue.append(s)
    return level


def build_groups(tasks: Dict) -> List[List[str]]:
    level  = compute_levels(tasks)
    max_lv = max(level.values())
    groups = [[] for _ in range(max_lv + 1)]
    for t, lv in level.items():
        groups[lv].append(t)
    for g in groups:
        g.sort()
    return groups


def groups_to_order(groups: List[List[str]]) -> List[str]:
    return [t for g in groups for t in g]

# ══════════════════════════════════════════════════════════════════════
#  3. ÉVALUATION
# ══════════════════════════════════════════════════════════════════════

def evaluate(order: List[str], vm_map: Dict,
             tasks: Dict, vms: Dict) -> Tuple:
    ft = {}; st = {}
    vm_avail = {vm: 0.0 for vm in vms}
    cost = energy = 0.0
    for task in order:
        vn  = vm_map[task]
        vm  = vms[vn]
        dur = tasks[task]["duration"] / vm["speed"]
        rdy = 0.0
        for dep in tasks[task]["deps"]:
            f = ft[dep]
            if vm_map[dep] != vn:
                f += tasks[task]["data"].get(dep, 0) * 0.01
            rdy = max(rdy, f)
        ts = max(rdy, vm_avail[vn])
        te = ts + dur
        st[task] = ts; ft[task] = te
        vm_avail[vn] = te
        cost   += dur / 3600 * vm["cost"]
        energy += dur * vm["power"] / 1000
    return (max(ft.values()), cost, energy,
            {"start_time": st, "finish_time": ft})

# ══════════════════════════════════════════════════════════════════════
#  4. NORMALISATION & POIDS DYNAMIQUES
# ══════════════════════════════════════════════════════════════════════

def compute_ref(tasks: Dict, vms: Dict,
                groups_ref: List, n_samples: int = 300) -> Dict:
    vm_names = list(vms.keys())
    all_m, all_c, all_e = [], [], []
    for vm in vm_names:
        order  = groups_to_order(groups_ref)
        vm_map = {t: vm for t in tasks}
        m, c, e, _ = evaluate(order, vm_map, tasks, vms)
        all_m.append(m); all_c.append(c); all_e.append(e)
    rng = random.Random(0)
    for _ in range(n_samples):
        rg     = [rng.sample(g, len(g)) for g in groups_ref]
        order  = groups_to_order(rg)
        vm_map = {t: rng.choice(vm_names) for t in tasks}
        m, c, e, _ = evaluate(order, vm_map, tasks, vms)
        all_m.append(m); all_c.append(c); all_e.append(e)
    ref = {
        "m_min": min(all_m), "m_max": max(all_m),
        "c_min": min(all_c), "c_max": max(all_c),
        "e_min": min(all_e), "e_max": max(all_e),
    }
    for k in ["m", "c", "e"]:
        if ref[f"{k}_max"] - ref[f"{k}_min"] < 1e-9:
            ref[f"{k}_max"] = ref[f"{k}_min"] + 1.0
    return ref


def normalize_obj(m: float, c: float, e: float, ref: Dict) -> Tuple:
    """Normalisation min-max → (0,1]."""
    f1 = (m - ref["m_min"]) / (ref["m_max"] - ref["m_min"]) + 1e-6
    f2 = (c - ref["c_min"]) / (ref["c_max"] - ref["c_min"]) + 1e-6
    f3 = (e - ref["e_min"]) / (ref["e_max"] - ref["e_min"]) + 1e-6
    return f1, f2, f3


def dynamic_weights(f1: float, f2: float, f3: float) -> Tuple:
    """
    Poids inversement proportionnels : wᵢ ∝ 1/fᵢ.
    w1=(f2·f3)/D, w2=(f1·f3)/D, w3=(f1·f2)/D, D=f1f2+f1f3+f2f3.
    """
    D = f1*f2 + f1*f3 + f2*f3 + 1e-9
    return (f2*f3)/D, (f1*f3)/D, (f1*f2)/D


def scalar_fitness(m: float, c: float, e: float, ref: Dict) -> float:
    f1, f2, f3 = normalize_obj(m, c, e, ref)
    w1, w2, w3 = dynamic_weights(f1, f2, f3)
    return w1*f1 + w2*f2 + w3*f3

# ══════════════════════════════════════════════════════════════════════
#  5. MÉTRIQUES : HV normalisé & IGD+
# ══════════════════════════════════════════════════════════════════════

def build_global_bounds(tasks: Dict, vms: Dict,
                        groups_ref: List,
                        n_samples: int = 500,
                        seed: int = 999) -> Dict:
    """Bornes globales FIXES indépendantes des algorithmes."""
    vm_names = list(vms.keys())
    rng      = random.Random(seed)
    all_m, all_c, all_e = [], [], []
    for vm in vm_names:
        order  = groups_to_order(groups_ref)
        vm_map = {t: vm for t in tasks}
        m, c, e, _ = evaluate(order, vm_map, tasks, vms)
        all_m.append(m); all_c.append(c); all_e.append(e)
    for _ in range(n_samples):
        rg     = [rng.sample(g, len(g)) for g in groups_ref]
        order  = groups_to_order(rg)
        vm_map = {t: rng.choice(vm_names) for t in tasks}
        m, c, e, _ = evaluate(order, vm_map, tasks, vms)
        all_m.append(m); all_c.append(c); all_e.append(e)
    bounds = {
        "m_min": min(all_m), "m_max": max(all_m),
        "c_min": min(all_c), "c_max": max(all_c),
        "e_min": min(all_e), "e_max": max(all_e),
    }
    for k in ["m", "c", "e"]:
        if bounds[f"{k}_max"] - bounds[f"{k}_min"] < 1e-9:
            bounds[f"{k}_max"] = bounds[f"{k}_min"] + 1.0
    return bounds


def normalize_to_unit(obj_list: List[Tuple], bounds: Dict) -> List[Tuple]:
    """Normalise (m,c,e) en [0,1]³ avec les bornes globales."""
    def _n(m, c, e):
        f1 = max(0.0, min(1.0, (m-bounds["m_min"])/(bounds["m_max"]-bounds["m_min"])))
        f2 = max(0.0, min(1.0, (c-bounds["c_min"])/(bounds["c_max"]-bounds["c_min"])))
        f3 = max(0.0, min(1.0, (e-bounds["e_min"])/(bounds["e_max"]-bounds["e_min"])))
        return (f1, f2, f3)
    return [_n(*o) for o in obj_list]


def _pareto_nd(pts: List[Tuple]) -> List[Tuple]:
    nd = []
    for p in pts:
        if not any(
            all(qi <= pi for qi, pi in zip(q, p)) and
            any(qi <  pi for qi, pi in zip(q, p))
            for q in pts if q is not p
        ):
            nd.append(p)
    return nd


def build_ref_front_pstar(tasks: Dict, vms: Dict,
                           groups_ref: List, bounds: Dict,
                           n_samples: int = 2000,
                           seed: int = 888) -> List[Tuple]:
    """
    Front P* ROBUSTE (normalisé) pour IGD+.

    Pourquoi pas uniquement l'échantillonnage aléatoire ?
    ───────────────────────────────────────────────────────
    Sur des workflows de taille modeste (≤ 50 tâches), les algorithmes
    évolutionnaires convergent souvent vers des solutions qui DOMINENT
    l'échantillonnage aléatoire : P*_aléatoire ⊂ front_algo, ce qui
    donne d⁺(a,b) = 0 pour tout b ∈ P* → IGD+ = 0.

    Solution : P* = union des fronts non-dominés issus de
    N_BOOTSTRAP runs indépendants des DEUX algorithmes +
    N_RAND solutions aléatoires → filtrage non-dominé global.
    Ce front représente la meilleure approximation connue de
    l'optimal, construite de façon équitable pour les deux algos.
    """
    vm_names   = list(vms.keys())
    rng        = random.Random(seed)
    all_pts    = []

    # ── Partie 1 : solutions aléatoires ───────────────────────
    n_rand = n_samples // 4
    for _ in range(n_rand):
        rg     = [rng.sample(g, len(g)) for g in groups_ref]
        order  = groups_to_order(rg)
        vm_map = {t: rng.choice(vm_names) for t in tasks}
        m, c, e, _ = evaluate(order, vm_map, tasks, vms)
        all_pts.append((m, c, e))

    # ── Partie 2 : bootstrap multi-runs des deux algos ────────
    # On importe dynamiquement les deux algos (définis plus bas)
    # pour construire un front de référence riche et équitable.
    # N_BOOTSTRAP runs courts suffisent.
    N_BOOTSTRAP = 8
    ref_internal = compute_ref(tasks, vms, groups_ref, n_samples=100)

    for r in range(N_BOOTSTRAP):
        s = seed + r * 31 + 1
        # NSGA-II Simple bootstrap — runs courts mais réels
        _, obj_s_full = nsga2_simple(
            tasks, vms, pop_size=40, n_gen=80,
            cx_rate=0.8, mut_rate=0.05,
            seed=s, verbose=False,
            ref=ref_internal)
        for o in obj_s_full:
            all_pts.append(o)

        # NSGA-II+HEFT bootstrap
        _, obj_h_full = nsga2_heft_hybrid(
            tasks, vms, pop_size=40, n_gen=80,
            cx_rate=0.8, mut_rate=0.05,
            heft_ratio=0.3, ls_rate=0.4,
            seed=s, verbose=False,
            ref=ref_internal)
        for o in obj_h_full:
            all_pts.append(o)

    # ── Filtrage global non-dominé + normalisation ─────────────
    all_norm = normalize_to_unit(all_pts, bounds)
    return _pareto_nd(all_norm)


def _hv2d(pts: List[Tuple], rx: float, ry: float) -> float:
    if not pts: return 0.0
    spts   = sorted(set(pts), key=lambda p: p[0])
    area   = 0.0
    prev_x = rx
    min_y  = ry
    for x, y in reversed(spts):
        if y < min_y:
            area  += (prev_x - x) * (min_y - y)
            prev_x = x
            min_y  = y
    area += (prev_x - spts[0][0]) * (min_y - spts[0][1])
    return max(area, 0.0)


def compute_hv(norm_obj: List[Tuple],
               ref_point: Tuple = (1.1, 1.1, 1.1)) -> float:
    """
    HV 3D normalisé ∈ [0, 1.331].
    Entrée : objectifs déjà normalisés en [0,1]³.
    ↑ Mieux.
    """
    ref = ref_point
    pts = [o for o in norm_obj
           if o[0] < ref[0] and o[1] < ref[1] and o[2] < ref[2]]
    if not pts: return 0.0
    pts = _pareto_nd(pts)
    pts_s  = sorted(pts, key=lambda p: p[2], reverse=True)
    prev_z = ref[2]
    hv     = 0.0
    active = []
    for p in pts_s:
        z_sl = prev_z - p[2]
        active.append((p[0], p[1]))
        area   = _hv2d(sorted(set(active), key=lambda a: a[0]),
                       ref[0], ref[1])
        hv    += area * z_sl
        prev_z = p[2]
    return hv


def compute_igd_plus(norm_front_A: List[Tuple],
                     norm_ref_B:   List[Tuple]) -> float:
    """
    IGD+(A,B) = (1/|B|)·Σ_{b∈B} min_{a∈A} d⁺(a,b)
    d⁺(a,b)  = √Σ max(0, aᵢ−bᵢ)²
    ↓ Mieux (0 = parfait).
    """
    if not norm_ref_B or not norm_front_A:
        return float("inf")
    total = 0.0
    for b in norm_ref_B:
        min_d = min(
            math.sqrt(sum(max(0.0, a[i]-b[i])**2 for i in range(3)))
            for a in norm_front_A
        )
        total += min_d
    return total / len(norm_ref_B)

# ══════════════════════════════════════════════════════════════════════
#  6. HEFT
# ══════════════════════════════════════════════════════════════════════

def heft_schedule(tasks: Dict, vms: Dict) -> Tuple:
    succ = {t: [] for t in tasks}
    for t in tasks:
        for dep in tasks[t]["deps"]:
            succ[dep].append(t)
    n      = len(vms)
    rank_u = {}
    def _r(t):
        if t in rank_u: return rank_u[t]
        w = sum(tasks[t]["duration"]/v["speed"] for v in vms.values()) / n
        if not succ[t]:
            rank_u[t] = w
        else:
            rank_u[t] = w + max(
                tasks[s]["data"].get(t,0)*0.01*(n-1)/n + _r(s)
                for s in succ[t])
        return rank_u[t]
    for t in tasks: _r(t)
    order    = sorted(tasks.keys(), key=lambda t: rank_u[t], reverse=True)
    vm_avail = {vm: 0.0 for vm in vms}
    ft = {}; st = {}; vm_map = {}
    for task in order:
        best_vm = None; best_eft = float("inf"); best_ast = 0.0
        for vn, vm in vms.items():
            dur = tasks[task]["duration"] / vm["speed"]
            rdy = 0.0
            for dep in tasks[task]["deps"]:
                f = ft[dep]
                if vm_map[dep] != vn:
                    f += tasks[task]["data"].get(dep,0)*0.01
                rdy = max(rdy, f)
            ast = max(rdy, vm_avail[vn])
            eft = ast + dur
            if eft < best_eft:
                best_eft, best_vm, best_ast = eft, vn, ast
        vm_map[task] = best_vm; st[task] = best_ast
        ft[task] = best_eft;    vm_avail[best_vm] = best_eft
    level  = compute_levels(tasks)
    max_lv = max(level.values())
    groups = [[] for _ in range(max_lv+1)]
    for t in order: groups[level[t]].append(t)
    return groups, vm_map


def critical_path(groups: List, vm_map: Dict,
                  tasks: Dict, vms: Dict) -> List[str]:
    order = groups_to_order(groups)
    _, _, _, sched = evaluate(order, vm_map, tasks, vms)
    ft_  = sched["finish_time"]; st_ = sched["start_time"]
    mksp = max(ft_.values())
    succ = {t: [] for t in tasks}
    for t in tasks:
        for dep in tasks[t]["deps"]: succ[dep].append(t)
    lft = {}
    for t in reversed(order):
        lft[t] = mksp if not succ[t] else min(st_[s] for s in succ[t])
    return [t for t in order if abs(lft[t]-ft_[t]) < 1e-6]


def heft_local_search(groups: List, vm_map: Dict,
                      tasks: Dict, vms: Dict) -> Tuple:
    cp     = set(critical_path(groups, vm_map, tasks, vms))
    new_vm = copy.deepcopy(vm_map)
    order  = groups_to_order(groups)
    idx_m  = {t: i for i,t in enumerate(order)}
    for task in order:
        if task not in cp: continue
        _, _, _, sched = evaluate(order, new_vm, tasks, vms)
        ft_ = sched["finish_time"]
        bv  = new_vm[task]; be = float("inf")
        for vn, vm in vms.items():
            dur = tasks[task]["duration"] / vm["speed"]
            rdy = 0.0
            for dep in tasks[task]["deps"]:
                f = ft_[dep]
                if new_vm[dep] != vn:
                    f += tasks[task]["data"].get(dep,0)*0.01
                rdy = max(rdy, f)
            busy = max(
                (ft_[t2] for t2 in order
                 if t2 != task and new_vm[t2]==vn and idx_m[t2]<idx_m[task]),
                default=0.0)
            eft = max(rdy, busy) + dur
            if eft < be: be, bv = eft, vn
        new_vm[task] = bv
    return groups, new_vm

# ══════════════════════════════════════════════════════════════════════
#  7. NOYAU NSGA-II
# ══════════════════════════════════════════════════════════════════════

def dominates(a: Tuple, b: Tuple) -> bool:
    return (all(x<=y for x,y in zip(a,b)) and
            any(x< y for x,y in zip(a,b)))


def fast_non_dominated_sort(obj: List[Tuple]) -> List[List[int]]:
    n   = len(obj)
    cnt = [0]*n; dom = [[] for _ in range(n)]; fronts = [[]]
    for i in range(n):
        for j in range(n):
            if i==j: continue
            if dominates(obj[i],obj[j]): dom[i].append(j)
            elif dominates(obj[j],obj[i]): cnt[i]+=1
        if cnt[i]==0: fronts[0].append(i)
    k=0
    while fronts[k]:
        nxt=[]
        for i in fronts[k]:
            for j in dom[i]:
                cnt[j]-=1
                if cnt[j]==0: nxt.append(j)
        k+=1; fronts.append(nxt)
    return [f for f in fronts if f]


def crowding_std(front: List[int], obj: List[Tuple]) -> Dict[int,float]:
    """Crowding distance standard (NSGA-II Simple)."""
    n    = len(front)
    dist = {i: 0.0 for i in front}
    if n <= 2:
        for i in front: dist[i] = float("inf")
        return dist
    for m in range(3):
        sf  = sorted(front, key=lambda i: obj[i][m])
        rng = obj[sf[-1]][m] - obj[sf[0]][m] + 1e-9
        dist[sf[0]] = dist[sf[-1]] = float("inf")
        for k in range(1,n-1):
            dist[sf[k]] += (obj[sf[k+1]][m]-obj[sf[k-1]][m])/rng
    return dist


def crowding_weighted(front: List[int], obj: List[Tuple],
                      ref: Dict) -> Dict[int,float]:
    """
    Crowding distance pondérée par poids dynamiques normalisés.
    Utilisée dans NSGA-II+HEFT Hybride.
    """
    n    = len(front)
    dist = {i: 0.0 for i in front}
    if n <= 2:
        for i in front: dist[i] = float("inf")
        return dist
    all_f  = [normalize_obj(*obj[i], ref) for i in front]
    avg_f1 = sum(f[0] for f in all_f)/n
    avg_f2 = sum(f[1] for f in all_f)/n
    avg_f3 = sum(f[2] for f in all_f)/n
    D      = avg_f1*avg_f2 + avg_f1*avg_f3 + avg_f2*avg_f3 + 1e-9
    ws     = [(avg_f2*avg_f3)/D, (avg_f1*avg_f3)/D, (avg_f1*avg_f2)/D]
    for m, w in enumerate(ws):
        sf  = sorted(front, key=lambda i: obj[i][m])
        rng = obj[sf[-1]][m] - obj[sf[0]][m] + 1e-9
        dist[sf[0]] = dist[sf[-1]] = float("inf")
        for k in range(1,n-1):
            dist[sf[k]] += w*(obj[sf[k+1]][m]-obj[sf[k-1]][m])/rng
    return dist


def binary_tournament(pop_size: int,
                       ranks: List[int],
                       dists: Dict) -> int:
    i, j = random.sample(range(pop_size), 2)
    if   ranks[i] < ranks[j]:                         return i
    elif ranks[j] < ranks[i]:                         return j
    elif dists.get(i,0) >= dists.get(j,0):            return i
    else:                                              return j


def env_select(pop: List, obj: List, size: int,
               crowd_fn, ref: Dict=None) -> Tuple:
    fronts  = fast_non_dominated_sort(obj)
    new_pop = []; new_obj = []
    for front in fronts:
        if len(new_pop)+len(front) <= size:
            for idx in front:
                new_pop.append(pop[idx]); new_obj.append(obj[idx])
        else:
            rem = size - len(new_pop)
            cd  = (crowd_fn(front,obj,ref) if ref is not None
                   else crowd_fn(front,obj))
            sf  = sorted(front, key=lambda i: cd.get(i,0), reverse=True)
            for idx in sf[:rem]:
                new_pop.append(pop[idx]); new_obj.append(obj[idx])
            break
    return new_pop, new_obj

# ══════════════════════════════════════════════════════════════════════
#  8. OPÉRATEURS — ENCODAGE PERMUTATION (NSGA-II Simple)
# ══════════════════════════════════════════════════════════════════════

def kahn_random(tasks: Dict) -> List[str]:
    """Ordre topologique aléatoire valide (Kahn shuffle)."""
    in_deg = {t: len(tasks[t]["deps"]) for t in tasks}
    succ   = {t: [] for t in tasks}
    for t in tasks:
        for dep in tasks[t]["deps"]: succ[dep].append(t)
    ready = [t for t in tasks if in_deg[t]==0]
    order = []
    while ready:
        t = random.choice(ready); ready.remove(t); order.append(t)
        for s in succ[t]:
            in_deg[s] -= 1
            if in_deg[s]==0: ready.append(s)
    return order


def ox_kahn(o1: List, o2: List, tasks: Dict) -> List:
    """Croisement OX biaisé sur ordres topologiques."""
    n    = len(o1)
    a, b = sorted(random.sample(range(n), 2))
    seg  = set(o1[a:b+1])
    prio = {}
    for rank, t in enumerate(o1):
        prio[t] = rank if t in seg else (
            n + next(i for i,x in enumerate(o2) if x==t))
    in_deg = {t: len(tasks[t]["deps"]) for t in tasks}
    succ   = {t: [] for t in tasks}
    for t in tasks:
        for dep in tasks[t]["deps"]: succ[dep].append(t)
    ready  = sorted([t for t in tasks if in_deg[t]==0],
                    key=lambda t: prio[t])
    result = []
    while ready:
        node = ready.pop(0); result.append(node)
        for s in succ[node]:
            in_deg[s] -= 1
            if in_deg[s]==0:
                ready.append(s); ready.sort(key=lambda x: prio[x])
    return result


def mutate_perm(order: List, tasks: Dict, rate: float) -> List:
    new = order[:]
    n   = len(new)
    for _ in range(n):
        if random.random() < rate:
            i, j = random.randint(0,n-1), random.randint(0,n-1)
            if i==j: continue
            new[i], new[j] = new[j], new[i]
            pos   = {t: k for k,t in enumerate(new)}
            valid = all(pos[dep] < pos[t]
                        for t in new for dep in tasks[t]["deps"])
            if not valid:
                new[i], new[j] = new[j], new[i]
    return new

# ══════════════════════════════════════════════════════════════════════
#  9. OPÉRATEURS — ENCODAGE TOPOLOGIQUE (NSGA-II+HEFT Hybride)
# ══════════════════════════════════════════════════════════════════════

def rand_chrom_topo(tasks: Dict, vms: Dict) -> Tuple:
    groups   = build_groups(tasks)
    vm_names = list(vms.keys())
    rg       = [random.sample(g, len(g)) for g in groups]
    vm_map   = {t: random.choice(vm_names) for t in tasks}
    return rg, vm_map


def _ox_level(a: List, b: List, cut: int) -> List:
    seg = a[:cut]
    return seg + [t for t in b if t not in set(seg)]


def crossover_topo(p1: Tuple, p2: Tuple) -> Tuple:
    """OX niveau par niveau + VM uniforme."""
    o1,v1 = p1; o2,v2 = p2
    c1g=[]; c2g=[]
    for g1,g2 in zip(o1,o2):
        n = len(g1)
        if n<=1: c1g.append(g1[:]); c2g.append(g2[:])
        else:
            pt = random.randint(1,n-1)
            c1g.append(_ox_level(g1,g2,pt))
            c2g.append(_ox_level(g2,g1,pt))
    c1v,c2v = {},{}
    for t in v1:
        if random.random()<0.5: c1v[t],c2v[t]=v1[t],v2[t]
        else:                   c1v[t],c2v[t]=v2[t],v1[t]
    return (c1g,c1v),(c2g,c2v)


def mutate_topo(chrom: Tuple, vms: Dict, rate: float) -> Tuple:
    """Swap intra-niveau + réassignation VM."""
    groups, vm_map = chrom
    vm_names = list(vms.keys())
    ng = []
    for g in groups:
        lg = g[:]
        if len(lg)>1:
            for i in range(len(lg)):
                if random.random()<rate:
                    j = random.randint(0,len(lg)-1)
                    lg[i],lg[j] = lg[j],lg[i]
        ng.append(lg)
    nv = copy.deepcopy(vm_map)
    for t in nv:
        if random.random()<rate:
            nv[t] = random.choice(vm_names)
    return ng, nv

# ══════════════════════════════════════════════════════════════════════
#  10. ALGO 1 — NSGA-II SIMPLE
#      Encodage permutation + normalisation + crowding standard
# ══════════════════════════════════════════════════════════════════════

def nsga2_simple(tasks: Dict, vms: Dict,
                 pop_size: int = 80, n_gen: int = 200,
                 cx_rate: float = 0.9, mut_rate: float = 0.05,
                 seed: int = 42, verbose: bool = True,
                 ref: Dict = None) -> Tuple:
    """
    NSGA-II Simple :
    • Encodage permutation (Kahn aléatoire)
    • Respect dépendances sans encodage topologique par niveaux
    • Normalisation min-max des objectifs
    • Crowding distance STANDARD (sans poids dynamiques)
    """
    random.seed(seed)
    vm_names   = list(vms.keys())
    groups_ref = build_groups(tasks)
    if ref is None:
        ref = compute_ref(tasks, vms, groups_ref)

    def rand_ch():
        order  = kahn_random(tasks)
        vm_map = {t: random.choice(vm_names) for t in tasks}
        return order, vm_map

    def get_obj(ch):
        order, vm_map = ch
        m,c,e,_ = evaluate(order, vm_map, tasks, vms)
        return (m,c,e)

    def cx(p1, p2):
        o1,v1 = p1; o2,v2 = p2
        if random.random() >= cx_rate:
            return copy.deepcopy(p1), copy.deepcopy(p2)
        co1 = ox_kahn(o1,o2,tasks)
        co2 = ox_kahn(o2,o1,tasks)
        cv1,cv2 = {},{}
        for t in v1:
            if random.random()<0.5: cv1[t],cv2[t]=v1[t],v2[t]
            else:                   cv1[t],cv2[t]=v2[t],v1[t]
        return (co1,cv1),(co2,cv2)

    def mut(ch):
        order, vm_map = ch
        new_o = mutate_perm(order, tasks, mut_rate)
        new_v = copy.deepcopy(vm_map)
        for t in new_v:
            if random.random()<mut_rate:
                new_v[t] = random.choice(vm_names)
        return new_o, new_v

    pop = [rand_ch() for _ in range(pop_size)]
    obj = [get_obj(p) for p in pop]

    if verbose:
        print(f"\n  NSGA-II Simple  {'Gen':>5}  {'|F0|':>6}  "
              f"{'MinMksp':>10}  {'MinCost':>10}  {'MinEnrg':>10}")
        print("  " + "─"*60)

    for gen in range(n_gen):
        fronts = fast_non_dominated_sort(obj)
        ranks  = [0]*pop_size; dists = {}
        for rank, front in enumerate(fronts):
            for idx in front: ranks[idx] = rank
            dists.update(crowding_std(front, obj))

        children=[]; ch_obj=[]
        while len(children)<pop_size:
            i1 = binary_tournament(pop_size, ranks, dists)
            i2 = binary_tournament(pop_size, ranks, dists)
            c1,c2 = cx(pop[i1], pop[i2])
            c1=mut(c1); c2=mut(c2)
            children.append(c1); ch_obj.append(get_obj(c1))
            if len(children)<pop_size:
                children.append(c2); ch_obj.append(get_obj(c2))

        pop,obj = env_select(pop+children, obj+ch_obj, pop_size,
                             crowding_std)

        if verbose and (gen%40==0 or gen==n_gen-1):
            pf = fronts[0]
            po = [obj[i] for i in pf]
            print(f"  NSGA-II Simple  {gen:>5}  {len(pf):>6}  "
                  f"{min(o[0] for o in po):>10.4f}  "
                  f"{min(o[1] for o in po):>10.6f}  "
                  f"{min(o[2] for o in po):>10.6f}")

    pf_idx = fast_non_dominated_sort(obj)[0]
    return [pop[i] for i in pf_idx], [obj[i] for i in pf_idx]

# ══════════════════════════════════════════════════════════════════════
#  11. ALGO 2 — NSGA-II + HEFT HYBRIDE
#      Encodage topologique par niveaux + normalisation + poids dynamiques
# ══════════════════════════════════════════════════════════════════════

def nsga2_heft_hybrid(tasks: Dict, vms: Dict,
                      pop_size: int = 80, n_gen: int = 200,
                      cx_rate: float = 0.9, mut_rate: float = 0.05,
                      heft_ratio: float = 0.3, ls_rate: float = 0.4,
                      seed: int = 42, verbose: bool = True,
                      ref: Dict = None) -> Tuple:
    """
    NSGA-II + HEFT Hybride (Notre méthode) :
    • Encodage topologique par niveaux → jamais invalide
    • Croisement OX niveau par niveau + VM uniforme
    • Mutation swap intra-niveau + réassignation VM
    • 1 HEFT exact + 30% HEFT perturbé + 70% aléatoire
    • Local search HEFT sur chemin critique (ls_rate=0.4)
    • Normalisation + poids dynamiques + crowding pondérée
    """
    random.seed(seed)
    vm_names   = list(vms.keys())
    groups_ref = build_groups(tasks)
    if ref is None:
        ref = compute_ref(tasks, vms, groups_ref)

    def get_obj(ch):
        g, vm = ch
        m,c,e,_ = evaluate(groups_to_order(g), vm, tasks, vms)
        return (m,c,e)

    # Population initiale hybride
    heft_g, heft_v = heft_schedule(tasks, vms)
    pop = [(heft_g, copy.deepcopy(heft_v))]
    n_heft = max(1, int(pop_size*heft_ratio))-1
    for _ in range(n_heft):
        pop.append(([g[:] for g in heft_g],
                    {t: random.choice(vm_names) for t in tasks}))
    while len(pop) < pop_size:
        pop.append(rand_chrom_topo(tasks, vms))

    obj = [get_obj(p) for p in pop]

    if verbose:
        print(f"\n  NSGA-II+HEFT    {'Gen':>5}  {'|F0|':>6}  "
              f"{'MinMksp':>10}  {'MinCost':>10}  {'MinEnrg':>10}")
        print("  " + "─"*60)

    for gen in range(n_gen):
        fronts = fast_non_dominated_sort(obj)
        ranks  = [0]*pop_size; dists = {}
        for rank, front in enumerate(fronts):
            for idx in front: ranks[idx] = rank
            dists.update(crowding_weighted(front, obj, ref))

        children=[]; ch_obj=[]
        while len(children)<pop_size:
            i1 = binary_tournament(pop_size, ranks, dists)
            i2 = binary_tournament(pop_size, ranks, dists)
            if random.random()<cx_rate:
                c1,c2 = crossover_topo(pop[i1], pop[i2])
            else:
                c1,c2 = copy.deepcopy(pop[i1]),copy.deepcopy(pop[i2])
            c1=mutate_topo(c1,vms,mut_rate)
            c2=mutate_topo(c2,vms,mut_rate)
            if random.random()<ls_rate:
                c1=heft_local_search(*c1,tasks,vms)
            if random.random()<ls_rate:
                c2=heft_local_search(*c2,tasks,vms)
            children.append(c1); ch_obj.append(get_obj(c1))
            if len(children)<pop_size:
                children.append(c2); ch_obj.append(get_obj(c2))

        pop,obj = env_select(pop+children, obj+ch_obj, pop_size,
                             crowding_weighted, ref=ref)

        if verbose and (gen%40==0 or gen==n_gen-1):
            pf = fronts[0]
            po = [obj[i] for i in pf]
            print(f"  NSGA-II+HEFT    {gen:>5}  {len(pf):>6}  "
                  f"{min(o[0] for o in po):>10.4f}  "
                  f"{min(o[1] for o in po):>10.6f}  "
                  f"{min(o[2] for o in po):>10.6f}")

    pf_idx = fast_non_dominated_sort(obj)[0]
    pf_ch  = [pop[i] for i in pf_idx]
    pf_obj = [obj[i] for i in pf_idx]
    # Convertir en (order, vm_map) pour cohérence
    pf_std = [(groups_to_order(g), v) for g,v in pf_ch]
    return pf_std, pf_obj

# ══════════════════════════════════════════════════════════════════════
#  12. GRAPHIQUES
# ══════════════════════════════════════════════════════════════════════



def repair_kahn(order: List[str], tasks: Dict) -> List[str]:
    """Répare un ordre topologique via Kahn avec priorité sur l'ordre donné."""
    priority = {t: i for i, t in enumerate(order)}
    in_deg = {t: len(tasks[t]["deps"]) for t in tasks}
    succ   = {t: [] for t in tasks}
    for t in tasks:
        for dep in tasks[t]["deps"]: succ[dep].append(t)
    ready  = sorted([t for t in tasks if in_deg[t] == 0],
                    key=lambda t: priority.get(t, 999))
    result = []
    while ready:
        node = ready.pop(0); result.append(node)
        for s in succ[node]:
            in_deg[s] -= 1
            if in_deg[s] == 0:
                ready.append(s)
                ready.sort(key=lambda x: priority.get(x, 999))
    return result



# ══════════════════════════════════════════════════════════════════════
#  ALGO 3 — MOEA/D (Tchebycheff)
#  Encodage identique à NSGA-II Simple : ordre Kahn + vm_map
#  Vecteurs de poids uniformes 3D, voisinage T, archive élaguée
# ══════════════════════════════════════════════════════════════════════

def _moead_weights(H: int) -> List:
    """Vecteurs de poids uniformes 3D. H=divisions → C(H+2,2) vecteurs."""
    W = []
    for i in range(H + 1):
        for j in range(H + 1 - i):
            k = H - i - j
            W.append((i/H, j/H, k/H))
    return W


def _moead_neighbours(W: List, T: int) -> List:
    """T voisins les plus proches par distance euclidienne."""
    N = len(W)
    return [sorted(range(N),
                   key=lambda j: sum((W[i][d]-W[j][d])**2 for d in range(3)))[:T]
            for i in range(N)]


def _trim_archive(arch_sol: List, arch_obj: List, max_size: int):
    """
    Élague l'archive à max_size solutions :
    1. Filtre non-dominé
    2. Si encore trop grand → crowding distance → tronque
    """
    if len(arch_obj) <= max_size:
        return arch_sol, arch_obj
    # Étape 1 : non-dominé
    nd_idx = []
    for i, p in enumerate(arch_obj):
        dom = any(
            all(arch_obj[j][d] <= p[d] for d in range(3)) and
            any(arch_obj[j][d] <  p[d] for d in range(3))
            for j in range(len(arch_obj)) if j != i
        )
        if not dom:
            nd_idx.append(i)
    nd_sol = [arch_sol[i] for i in nd_idx]
    nd_obj = [arch_obj[i] for i in nd_idx]
    if len(nd_obj) <= max_size:
        return nd_sol, nd_obj
    # Étape 2 : crowding distance
    n = len(nd_obj); cd = [0.0] * n
    for m in range(3):
        idx_s = sorted(range(n), key=lambda i: nd_obj[i][m])
        cd[idx_s[0]] = cd[idx_s[-1]] = float("inf")
        lo = nd_obj[idx_s[0]][m]; hi = nd_obj[idx_s[-1]][m]
        rng = hi - lo + 1e-9
        for k in range(1, n - 1):
            cd[idx_s[k]] += (nd_obj[idx_s[k+1]][m] - nd_obj[idx_s[k-1]][m]) / rng
    kept = sorted(sorted(range(n), key=lambda i: cd[i], reverse=True)[:max_size])
    return [nd_sol[i] for i in kept], [nd_obj[i] for i in kept]


def moead(tasks: Dict, vms: Dict,
          H: int = 8, T: int = 5,
          n_gen: int = 200, mut_rate: float = 0.05,
          max_archive: int = None,
          seed: int = 42, verbose: bool = True) -> Tuple:
    """
    MOEA/D Tchebycheff :
      g(x|λ,z*) = max_i { λ_i · |f_i(x) − z*_i| }

    max_archive : taille max de l'archive (None=illimitée).
                  Passer pop_size des GA pour comparaison équitable.
    Retourne (pf_chroms, pf_obj) — même format que nsga2_simple.
    """
    random.seed(seed)
    vm_names   = list(vms.keys())
    groups_ref = build_groups(tasks)
    W = _moead_weights(H); N = len(W); B = _moead_neighbours(W, T)

    # ── Opérateurs ────────────────────────────────────────────────────
    def rand_sol():
        order  = kahn_random(tasks)
        vm_map = {t: random.choice(vm_names) for t in tasks}
        return order, vm_map

    def crossover_m(s1, s2):
        o1, vm1 = s1; o2, vm2 = s2
        n = len(o1); pt = random.randint(1, n - 2)
        seg = o1[:pt]; seg_s = set(seg)
        child_order = seg + [t for t in o2 if t not in seg_s]
        # repair topologique
        child_order = repair_kahn(child_order, tasks)
        child_vm = {t: (vm1[t] if random.random() < 0.5 else vm2[t])
                    for t in tasks}
        return child_order, child_vm

    def mutate_m(sol):
        order, vm_map = list(sol[0]), dict(sol[1])
        order = mutate_perm(order, tasks, mut_rate)
        for t in list(vm_map.keys()):
            if random.random() < mut_rate:
                vm_map[t] = random.choice(vm_names)
        return order, vm_map

    def tcheby(obj, w, z):
        return max(w[d] * abs(obj[d] - z[d]) for d in range(3))

    # ── Initialisation ────────────────────────────────────────────────
    pop  = [rand_sol() for _ in range(N)]
    objs = []
    for s in pop:
        m, c, e, _ = evaluate(s[0], s[1], tasks, vms)
        objs.append((m, c, e))

    z_star = [min(objs[i][d] for i in range(N)) for d in range(3)]

    # Archive externe non-dominée
    arch_sol = list(pop)
    arch_obj = list(objs)

    def update_archive(new_sol, new_obj):
        to_remove = []
        for k, ao in enumerate(arch_obj):
            if (all(new_obj[d] <= ao[d] for d in range(3)) and
                any(new_obj[d] <  ao[d] for d in range(3))):
                to_remove.append(k)
            elif (all(ao[d] <= new_obj[d] for d in range(3)) and
                  any(ao[d] <  new_obj[d] for d in range(3))):
                return
        for k in sorted(to_remove, reverse=True):
            arch_sol.pop(k); arch_obj.pop(k)
        arch_sol.append(new_sol); arch_obj.append(new_obj)

    if verbose:
        print(f"\n  MOEA/D          {'Gen':>5}  {'|Arch|':>7}  "
              f"{'z*[0]':>10}  {'z*[1]':>10}  {'z*[2]':>10}")
        print("  " + "─"*60)

    # ── Boucle principale ─────────────────────────────────────────────
    for gen in range(n_gen):
        for i in range(N):
            p1i, p2i = random.sample(B[i], 2)
            child = crossover_m(pop[p1i], pop[p2i])
            child = mutate_m(child)
            m, c, e, _ = evaluate(child[0], child[1], tasks, vms)
            c_obj = (m, c, e)
            # Mise à jour z*
            for d in range(3):
                if c_obj[d] < z_star[d]: z_star[d] = c_obj[d]
            # Mise à jour voisins
            for j in B[i]:
                if tcheby(c_obj, W[j], z_star) <= tcheby(objs[j], W[j], z_star):
                    pop[j] = child; objs[j] = c_obj
            update_archive(child, c_obj)

        # Élagage périodique
        if max_archive and len(arch_obj) > max_archive * 2:
            arch_sol[:], arch_obj[:] = _trim_archive(arch_sol, arch_obj, max_archive)

        if verbose and (gen % 40 == 0 or gen == n_gen - 1):
            print(f"  MOEA/D          {gen:>5}  {len(arch_obj):>7}  "
                  f"{z_star[0]:>10.4f}  {z_star[1]:>10.6f}  {z_star[2]:>10.6f}")

    # Élagage final
    if max_archive:
        arch_sol, arch_obj = _trim_archive(arch_sol, arch_obj, max_archive)

    return arch_sol, arch_obj



def plot_pareto_fronts_3(obj1: List[Tuple], obj2: List[Tuple], obj3: List[Tuple],
                          label1: str = "NSGA-II Simple",
                          label2: str = "NSGA-II+HEFT",
                          label3: str = "MOEA/D",
                          save_path: str = "/mnt/user-data/outputs/pareto_workflow10.png"):
    m1=[o[0] for o in obj1]; c1=[o[1] for o in obj1]; e1=[o[2] for o in obj1]
    m2=[o[0] for o in obj2]; c2=[o[1] for o in obj2]; e2=[o[2] for o in obj2]
    m3=[o[0] for o in obj3]; c3=[o[1] for o in obj3]; e3=[o[2] for o in obj3]

    fig = plt.figure(figsize=(16, 13))
    fig.suptitle(
        f"Fronts de Pareto — {label1}  vs  {label2}  vs  {label3}\n"
        "Workflow 30 tâches × 5 VMs",
        fontsize=13, fontweight="bold", y=0.98)

    def plot_2d(ax, configs):
        for (x,y,col,mk,lbl,cnt) in configs:
            ax.scatter(x, y, c=col, marker=mk, s=70, alpha=0.85,
                       label=f"{lbl} ({cnt} pts)", zorder=3)
            if x:
                s = sorted(zip(x, y))
                ax.plot([p[0] for p in s],[p[1] for p in s],
                        col, lw=0.9, alpha=0.35)

    ax1 = fig.add_subplot(2, 2, 1)
    plot_2d(ax1, [(m1,c1,"royalblue","o",label1,len(obj1)),
                  (m2,c2,"tomato","^",label2,len(obj2)),
                  (m3,c3,"#1D9E75","s",label3,len(obj3))])
    ax1.set_xlabel("Makespan (s)",fontsize=10); ax1.set_ylabel("Coût ($)",fontsize=10)
    ax1.set_title("Makespan vs Coût",fontsize=11,fontweight="bold")
    ax1.legend(fontsize=9); ax1.grid(True,alpha=0.3)

    ax2 = fig.add_subplot(2, 2, 2)
    plot_2d(ax2, [(m1,e1,"royalblue","o",label1,len(obj1)),
                  (m2,e2,"tomato","^",label2,len(obj2)),
                  (m3,e3,"#1D9E75","s",label3,len(obj3))])
    ax2.set_xlabel("Makespan (s)",fontsize=10); ax2.set_ylabel("Énergie (kj)",fontsize=10)
    ax2.set_title("Makespan vs Énergie",fontsize=11,fontweight="bold")
    ax2.legend(fontsize=9); ax2.grid(True,alpha=0.3)

    ax3 = fig.add_subplot(2, 2, 3)
    plot_2d(ax3, [(c1,e1,"royalblue","o",label1,len(obj1)),
                  (c2,e2,"tomato","^",label2,len(obj2)),
                  (c3,e3,"#1D9E75","s",label3,len(obj3))])
    ax3.set_xlabel("Coût ($)",fontsize=10); ax3.set_ylabel("Énergie (kj)",fontsize=10)
    ax3.set_title("Coût vs Énergie",fontsize=11,fontweight="bold")
    ax3.legend(fontsize=9); ax3.grid(True,alpha=0.3)

    ax4 = fig.add_subplot(2, 2, 4, projection="3d")
    ax4.scatter(m1,c1,e1,c="royalblue",marker="o",s=40,alpha=0.8,label=label1)
    ax4.scatter(m2,c2,e2,c="tomato",   marker="^",s=40,alpha=0.8,label=label2)
    ax4.scatter(m3,c3,e3,c="#1D9E75",  marker="s",s=40,alpha=0.8,label=label3)
    ax4.set_xlabel("Makespan (s)",fontsize=9)
    ax4.set_ylabel("Coût ($)",fontsize=9)
    ax4.set_zlabel("Énergie (kj)",fontsize=9)
    ax4.set_title("Vue 3D",fontsize=11,fontweight="bold")
    ax4.legend(fontsize=9)

    plt.tight_layout(rect=[0,0,1,0.96])
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Pareto sauvegardé : {save_path}")


def plot_metrics_bar_3(metrics: Dict,
                        save_path: str = "metrics_bar_workflow30.png"):
    labels  = list(metrics.keys())
    m_names = ["HV (↑ mieux)", "IGD+ (↓ mieux)", "|Pareto|"]
    colors  = ["royalblue", "tomato", "#1D9E75"]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, mn in zip(axes, m_names):
        idx  = m_names.index(mn)
        vals = [metrics[lab][idx] for lab in labels]
        bars = ax.bar(labels, vals, color=colors[:len(labels)],
                      alpha=0.85, edgecolor="black", width=0.45)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2,
                    bar.get_height()+max(abs(v) for v in vals)*0.01,
                    f"{val:.5f}", ha="center", va="bottom",
                    fontsize=10, fontweight="bold")
        ax.set_title(mn, fontsize=11, fontweight="bold")
        ax.set_ylabel("Valeur moyenne (20 runs)")
        ax.grid(axis="y", alpha=0.3)

    fig.suptitle("Métriques Pareto — Moyenne 20 runs\nWorkflow 30 tâches × 5 VMs",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Métriques sauvegardées : {save_path}")


def plot_convergence_runs_3(ga_hvs, hb_hvs, md_hvs,
                             ga_igds, hb_igds, md_igds,
                             save_path="runs_workflow30.png"):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    runs = list(range(1, len(ga_hvs)+1))
    avg  = lambda l: sum(l)/len(l)

    for vals, col, mk, lbl in [
        (ga_hvs, "royalblue", "o", "NSGA-II Simple"),
        (hb_hvs, "tomato",    "^", "NSGA-II+HEFT"),
        (md_hvs, "#1D9E75",   "s", "MOEA/D"),
    ]:
        axes[0].plot(runs, vals, f"{mk}-", c=col, lw=1.8,
                     label=f"{lbl} (moy={avg(vals):.4f})", markersize=5)
        axes[0].axhline(avg(vals), c=col, lw=1, ls="--", alpha=0.6)

    axes[0].set_title("HV par run (↑ mieux)", fontweight="bold")
    axes[0].set_xlabel("Run"); axes[0].set_ylabel("HV normalisé")
    axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

    for vals, col, mk, lbl in [
        (ga_igds, "royalblue", "o", "NSGA-II Simple"),
        (hb_igds, "tomato",    "^", "NSGA-II+HEFT"),
        (md_igds, "#1D9E75",   "s", "MOEA/D"),
    ]:
        axes[1].plot(runs, vals, f"{mk}-", c=col, lw=1.8,
                     label=f"{lbl} (moy={avg(vals):.5f})", markersize=5)
        axes[1].axhline(avg(vals), c=col, lw=1, ls="--", alpha=0.6)

    axes[1].set_title("IGD+ par run (↓ mieux)", fontweight="bold")
    axes[1].set_xlabel("Run"); axes[1].set_ylabel("IGD+")
    axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

    fig.suptitle("Évolution des métriques sur 20 runs indépendants\n"
                 "Workflow 30 tâches × 5 VMs",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Évolution runs sauvegardée : {save_path}")

def plot_pareto_fronts(obj1: List[Tuple], obj2: List[Tuple],
                       label1: str = "NSGA-II Simple",
                       label2: str = "NSGA-II+HEFT",
                       save_path: str = "pareto_workflow30.png"):
    m1=[o[0] for o in obj1]; c1=[o[1] for o in obj1]; e1=[o[2] for o in obj1]
    m2=[o[0] for o in obj2]; c2=[o[1] for o in obj2]; e2=[o[2] for o in obj2]

    fig = plt.figure(figsize=(16, 13))
    fig.suptitle(
        f"Fronts de Pareto — {label1}  vs  {label2}\n"
        "Workflow 30 tâches × 5 VMs",
        fontsize=13, fontweight="bold", y=0.98)

    def plot_2d(ax, x1, y1, x2, y2, xl, yl, title):
        ax.scatter(x1,y1, c="royalblue",marker="o",s=80,alpha=0.85,
                   label=f"{label1} ({len(obj1)} pts)", zorder=3)
        ax.scatter(x2,y2, c="tomato",   marker="^",s=80,alpha=0.85,
                   label=f"{label2} ({len(obj2)} pts)",  zorder=3)
        if x1:
            s=sorted(zip(x1,y1))
            ax.plot([p[0] for p in s],[p[1] for p in s],
                    "royalblue",lw=1.2,alpha=0.5)
        if x2:
            s=sorted(zip(x2,y2))
            ax.plot([p[0] for p in s],[p[1] for p in s],
                    "tomato",lw=1.2,alpha=0.5)
        ax.set_xlabel(xl,fontsize=10); ax.set_ylabel(yl,fontsize=10)
        ax.set_title(title,fontsize=11,fontweight="bold")
        ax.legend(fontsize=9); ax.grid(True,alpha=0.3)

    ax1=fig.add_subplot(2,2,1)
    plot_2d(ax1,m1,c1,m2,c2,"Makespan (s)","Coût ($)","Makespan vs Coût")

    ax2=fig.add_subplot(2,2,2)
    plot_2d(ax2,m1,e1,m2,e2,"Makespan (s)","Énergie (kj)","Makespan vs Énergie")

    ax3=fig.add_subplot(2,2,3)
    plot_2d(ax3,c1,e1,c2,e2,"Coût ($)","Énergie (kj)","Coût vs Énergie")

    ax4=fig.add_subplot(2,2,4,projection="3d")
    ax4.scatter(m1,c1,e1,c="royalblue",marker="o",s=50,alpha=0.8,label=label1)
    ax4.scatter(m2,c2,e2,c="tomato",   marker="^",s=50,alpha=0.8,label=label2)
    ax4.set_xlabel("Makespan (s)",fontsize=9)
    ax4.set_ylabel("Coût ($)",fontsize=9)
    ax4.set_zlabel("Énergie (kj)",fontsize=9)
    ax4.set_title("Vue 3D",fontsize=11,fontweight="bold")
    ax4.legend(fontsize=9)

    plt.tight_layout(rect=[0,0,1,0.96])
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Pareto sauvegardé : {save_path}")


def plot_metrics_bar(metrics: Dict,
                     save_path: str = "metrics_bar_workflow30.png"):
    labels = list(metrics.keys())
    m_names = ["HV (↑ mieux)", "IGD+ (↓ mieux)", "|Pareto|"]
    colors  = ["royalblue", "tomato"]

    fig, axes = plt.subplots(1,3,figsize=(14,5))
    for ax, mn in zip(axes, m_names):
        idx  = m_names.index(mn)
        vals = [metrics[lab][idx] for lab in labels]
        bars = ax.bar(labels, vals, color=colors[:len(labels)],
                      alpha=0.85, edgecolor="black", width=0.45)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2,
                    bar.get_height()+max(vals)*0.01,
                    f"{val:.5f}", ha="center", va="bottom",
                    fontsize=10, fontweight="bold")
        ax.set_title(mn, fontsize=11, fontweight="bold")
        ax.set_ylabel("Valeur moyenne (20 runs)")
        ax.grid(axis="y", alpha=0.3)

    fig.suptitle("Métriques Pareto — Moyenne 20 runs\nWorkflow 30 tâches × 5 VMs",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Métriques sauvegardées : {save_path}")


def plot_convergence_runs(ga_hvs, hb_hvs, ga_igds, hb_igds,
                          save_path="runs_workflow30.png"):
    fig, axes = plt.subplots(1,2,figsize=(13,5))
    runs = list(range(1, len(ga_hvs)+1))

    axes[0].plot(runs, ga_hvs, "o-", c="royalblue", lw=1.8,
                 label="NSGA-II Simple", markersize=5)
    axes[0].plot(runs, hb_hvs, "^-", c="tomato", lw=1.8,
                 label="NSGA-II+HEFT", markersize=5)
    axes[0].axhline(sum(ga_hvs)/len(ga_hvs), c="royalblue",
                    lw=1, ls="--", alpha=0.6)
    axes[0].axhline(sum(hb_hvs)/len(hb_hvs), c="tomato",
                    lw=1, ls="--", alpha=0.6)
    axes[0].set_title("HV par run (↑ mieux)", fontweight="bold")
    axes[0].set_xlabel("Run"); axes[0].set_ylabel("HV normalisé")
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(runs, ga_igds, "o-", c="royalblue", lw=1.8,
                 label="NSGA-II Simple", markersize=5)
    axes[1].plot(runs, hb_igds, "^-", c="tomato", lw=1.8,
                 label="NSGA-II+HEFT", markersize=5)
    axes[1].axhline(sum(ga_igds)/len(ga_igds), c="royalblue",
                    lw=1, ls="--", alpha=0.6)
    axes[1].axhline(sum(hb_igds)/len(hb_igds), c="tomato",
                    lw=1, ls="--", alpha=0.6)
    axes[1].set_title("IGD+ par run (↓ mieux)", fontweight="bold")
    axes[1].set_xlabel("Run"); axes[1].set_ylabel("IGD+")
    axes[1].legend(); axes[1].grid(alpha=0.3)

    fig.suptitle("Évolution des métriques sur 20 runs indépendants\n"
                 "Workflow 30 tâches × 5 VMs",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Évolution runs sauvegardée : {save_path}")

# ══════════════════════════════════════════════════════════════════════
#  13. AFFICHAGE TEXTUEL
# ══════════════════════════════════════════════════════════════════════

def display_pareto(label: str, pf_chroms: List, pf_obj: List,
                   tasks: Dict, vms: Dict, max_show: int = 6):
    W = 72
    print(f"\n  {'═'*W}")
    print(f"  FRONT DE PARETO — {label}  ({len(pf_obj)} solutions)")
    print(f"  {'═'*W}")
    print(f"  {'#':>4}  {'Makespan (s)':>13}  {'Coût ($)':>11}  "
          f"{'Énergie (kWh)':>14}  Profil")
    print(f"  {'─'*W}")
    min_m = min(o[0] for o in pf_obj)
    min_c = min(o[1] for o in pf_obj)
    min_e = min(o[2] for o in pf_obj)
    si    = sorted(range(len(pf_obj)), key=lambda i: pf_obj[i][0])
    for rank, i in enumerate(si[:max_show]):
        m,c,e = pf_obj[i]
        tags  = []
        if abs(m-min_m)<1e-4: tags.append("★makespan")
        if abs(c-min_c)<1e-9: tags.append("★coût")
        if abs(e-min_e)<1e-4: tags.append("★énergie")
        print(f"  {rank+1:>4}  {m:>13.4f}  {c:>11.6f}  "
              f"{e:>14.6f}  {'  '.join(tags)}")
    if len(pf_obj) > max_show:
        print(f"  ... {len(pf_obj)-max_show} autres solutions")
    print(f"  {'═'*W}")


def display_metrics_table(label1: str, hv1: float, igd1: float, nf1: int,
                          label2: str, hv2: float, igd2: float, nf2: int,
                          n_pstar: int):
    W = 72
    print("\n" + "═"*W)
    print("  MÉTRIQUES D'ÉVALUATION PARETO")
    print("─"*W)
    print(f"  Point de référence HV  : (1.1, 1.1, 1.1) — espace normalisé fixe")
    print(f"  Front P* (IGD+)        : {n_pstar} solutions indépendantes")
    print(f"  HV ∈ [0, 1.331]  — normalisé, comparaison équitable")
    print("─"*W)
    print(f"  {'Métrique':<28} {label1:>18}  {label2:>18}")
    print("─"*W)

    def w(v1,v2,hi=True):
        b1 = "◄" if (v1>v2 if hi else v1<v2) else "  "
        b2 = "◄" if (v2>v1 if hi else v2<v1) else "  "
        return b1, b2

    b1,b2 = w(hv1,hv2,hi=True)
    print(f"  {'HV (↑ mieux)':<28} {hv1:>16.6f}{b1}  {hv2:>16.6f}{b2}")
    b1,b2 = w(igd1,igd2,hi=False)
    print(f"  {'IGD+ (↓ mieux)':<28} {igd1:>16.6f}{b1}  {igd2:>16.6f}{b2}")
    b1,b2 = w(nf1,nf2,hi=True)
    print(f"  {'|Pareto| (↑ mieux)':<28} {nf1:>18}{b1}  {nf2:>18}{b2}")
    print("═"*W)
    return hv1, hv2, igd1, igd2

# ══════════════════════════════════════════════════════════════════════
#  14. MAIN
# ══════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    OUT    = "/mnt/user-data/outputs"
    tasks  = workflow_10()
    vms    = vms_5()
    N_RUNS = 20

    # ── Titre ─────────────────────────────────────────────────────────
    print("╔" + "═"*70 + "╗")
    print("║  NSGA-II Simple  vs  NSGA-II+HEFT Hybride" + " "*27 + "║")
    print(f"║  Workflow 30 tâches × {len(vms)} VMs  |  {N_RUNS} runs  |  seed=42" + " "*19 + "║")
    print("╚" + "═"*70 + "╝")

    groups_ref = build_groups(tasks)
    print(f"\n  Niveaux topologiques :")
    for i,g in enumerate(groups_ref):
        print(f"    Niveau {i} ({len(g):>2} tâches) : {g[:4]}"
              f"{'...' if len(g)>4 else ''}")

    # ── Bornes globales fixes ─────────────────────────────────────────
    print("\n  Calcul des bornes globales (500 éch., seed=999)...")
    bounds = build_global_bounds(tasks, vms, groups_ref,
                                  n_samples=500, seed=999)
    print(f"  Makespan : [{bounds['m_min']:.4f}, {bounds['m_max']:.4f}] s")
    print(f"  Coût     : [{bounds['c_min']:.6f}, {bounds['c_max']:.6f}] $")
    print(f"  Énergie  : [{bounds['e_min']:.6f}, {bounds['e_max']:.6f}] kWh")

    # ── Front P* indépendant ──────────────────────────────────────────
    print("\n  Construction du front P* (2000 éch., seed=888)...")
    pstar = build_ref_front_pstar(tasks, vms, groups_ref, bounds,
                                   n_samples=2000, seed=888)
    print(f"  P* : {len(pstar)} solutions non-dominées (normalisé [0,1]³)")

    # Référence pour normalisation interne aux algos
    ref_algo = compute_ref(tasks, vms, groups_ref, n_samples=300)

    # ════════════════════════════════════════════════════════════════
    #  20 RUNS
    # ════════════════════════════════════════════════════════════════
    ga_hvs,  ga_igds,  ga_nf  = [], [], []
    hb_hvs,  hb_igds,  hb_nf  = [], [], []
    md_hvs,  md_igds,  md_nf  = [], [], []

    print("\n" + "═"*76)
    print(f"  {N_RUNS} RUNS — NSGA-II Simple  vs  NSGA-II+HEFT")
    print(f"  HV normalisé ∈ [0,1.331]  |  IGD+ sur P* indépendant")
    print("═"*76)
    print(f"  {'Run':>4}  "
          f"{'GA_HV':>10} {'GA_IGD+':>10} {'GA|F0|':>7}  "
          f"{'HB_HV':>10} {'HB_IGD+':>10} {'HB|F0|':>7}  "
          f"{'MD_HV':>10} {'MD_IGD+':>10} {'MD|F0|':>7}")
    print("  " + "─"*90)

    for run in range(N_RUNS):
        # NSGA-II Simple
        pf1c, pf1o = nsga2_simple(
            tasks, vms, pop_size=70, n_gen=300,
            cx_rate=0.8, mut_rate=0.05,
            seed=run, verbose=False, ref=ref_algo)

        # NSGA-II+HEFT Hybride
        pf2c, pf2o = nsga2_heft_hybrid(
            tasks, vms, pop_size=70, n_gen=300,
            cx_rate=0.8, mut_rate=0.05,
            heft_ratio=0.3, ls_rate=0.4,
            seed=run, verbose=False, ref=ref_algo)

        # Métriques sur bornes fixes
        norm1 = normalize_to_unit(pf1o, bounds)
        norm2 = normalize_to_unit(pf2o, bounds)
        hv1   = compute_hv(norm1)
        hv2   = compute_hv(norm2)
        igd1  = compute_igd_plus(norm1, pstar)
        igd2  = compute_igd_plus(norm2, pstar)

        # MOEA/D
        pf3c, pf3o = moead(
            tasks, vms, H=8, T=5, n_gen=300,
            mut_rate=0.05, max_archive=70,
            seed=run, verbose=False)

        norm3 = normalize_to_unit(pf3o, bounds)
        hv3   = compute_hv(norm3)
        igd3  = compute_igd_plus(norm3, pstar)

        ga_hvs.append(hv1);  ga_igds.append(igd1);  ga_nf.append(len(pf1o))
        hb_hvs.append(hv2);  hb_igds.append(igd2);  hb_nf.append(len(pf2o))
        md_hvs.append(hv3);  md_igds.append(igd3);  md_nf.append(len(pf3o))

        print(f"  {run+1:>4}  "
              f"{hv1:>10.6f} {igd1:>10.6f} {len(pf1o):>7}  "
              f"{hv2:>10.6f} {igd2:>10.6f} {len(pf2o):>7}  "
              f"{hv3:>10.6f} {igd3:>10.6f} {len(pf3o):>7}")

    avg = lambda l: sum(l)/len(l)
    std = lambda l,m: math.sqrt(sum((x-m)**2 for x in l)/len(l))
    print("  " + "─"*90)
    print(f"  {'Moy':>4}  "
          f"{avg(ga_hvs):>10.6f} {avg(ga_igds):>10.6f} {avg(ga_nf):>7.1f}  "
          f"{avg(hb_hvs):>10.6f} {avg(hb_igds):>10.6f} {avg(hb_nf):>7.1f}  "
          f"{avg(md_hvs):>10.6f} {avg(md_igds):>10.6f} {avg(md_nf):>7.1f}")
    print(f"  {'Std':>4}  "
          f"{std(ga_hvs,avg(ga_hvs)):>10.6f} {std(ga_igds,avg(ga_igds)):>10.6f} {'—':>7}  "
          f"{std(hb_hvs,avg(hb_hvs)):>10.6f} {std(hb_igds,avg(hb_igds)):>10.6f} {'—':>7}  "
          f"{std(md_hvs,avg(md_hvs)):>10.6f} {std(md_igds,avg(md_igds)):>10.6f} {'—':>7}")
    print(f"  {'Min':>4}  "
          f"{min(ga_hvs):>10.6f} {min(ga_igds):>10.6f} {min(ga_nf):>7}  "
          f"{min(hb_hvs):>10.6f} {min(hb_igds):>10.6f} {min(hb_nf):>7}  "
          f"{min(md_hvs):>10.6f} {min(md_igds):>10.6f} {min(md_nf):>7}")
    print(f"  {'Max':>4}  "
          f"{max(ga_hvs):>10.6f} {max(ga_igds):>10.6f} {max(ga_nf):>7}  "
          f"{max(hb_hvs):>10.6f} {max(hb_igds):>10.6f} {max(hb_nf):>7}  "
          f"{max(md_hvs):>10.6f} {max(md_igds):>10.6f} {max(md_nf):>7}")
    print("═"*90)

    # ── Résumé statistique ────────────────────────────────────────────
    print("\n" + "═"*76)
    print("  RÉSUMÉ STATISTIQUE — 20 RUNS")
    print("═"*76)
    print(f"  {'Algorithme':<26} {'HV moy':>10} {'HV std':>8} "
          f"{'IGD+ moy':>10} {'IGD+ std':>8} {'|F0| moy':>9}")
    print("─"*76)
    print(f"  {'NSGA-II Simple':<26} "
          f"{avg(ga_hvs):>10.6f} {std(ga_hvs,avg(ga_hvs)):>8.6f} "
          f"{avg(ga_igds):>10.6f} {std(ga_igds,avg(ga_igds)):>8.6f} "
          f"{avg(ga_nf):>9.1f}")
    print(f"  {'NSGA-II+HEFT Hybride':<26} "
          f"{avg(hb_hvs):>10.6f} {std(hb_hvs,avg(hb_hvs)):>8.6f} "
          f"{avg(hb_igds):>10.6f} {std(hb_igds,avg(hb_igds)):>8.6f} "
          f"{avg(hb_nf):>9.1f}")
    print(f"  {'MOEA/D':<26} "
          f"{avg(md_hvs):>10.6f} {std(md_hvs,avg(md_hvs)):>8.6f} "
          f"{avg(md_igds):>10.6f} {std(md_igds,avg(md_igds)):>8.6f} "
          f"{avg(md_nf):>9.1f}")
    print("─"*76)
    dhv_hm  = avg(hb_hvs) - avg(ga_hvs)
    digd_hm = avg(hb_igds) - avg(ga_igds)
    dhv_md  = avg(md_hvs) - avg(ga_hvs)
    digd_md = avg(md_igds) - avg(ga_igds)
    print(f"  ΔHV   (HEFT−Simple)  = {dhv_hm:+.6f}  "
          f"→ {'HEFT meilleur ✓' if dhv_hm>0 else 'Simple meilleur'}")
    print(f"  ΔHV   (MOEA/D−Simple)= {dhv_md:+.6f}  "
          f"→ {'MOEA/D meilleur ✓' if dhv_md>0 else 'Simple meilleur'}")
    print(f"  ΔIGD+ (HEFT−Simple)  = {digd_hm:+.6f}  "
          f"→ {'HEFT meilleur ✓' if digd_hm<0 else 'Simple meilleur'}")
    print(f"  ΔIGD+ (MOEA/D−Simple)= {digd_md:+.6f}  "
          f"→ {'MOEA/D meilleur ✓' if digd_md<0 else 'Simple meilleur'}")
    print("═"*76)

    # ════════════════════════════════════════════════════════════════
    #  RUN PRINCIPAL seed=42
    # ════════════════════════════════════════════════════════════════
    print("\n\n" + "╔" + "═"*70 + "╗")
    print("║  RUN PRINCIPAL — seed=42" + " "*45 + "║")
    print("╚" + "═"*70 + "╝")

    print("\n  [1/3] NSGA-II Simple (seed=42, n_gen=300)...")
    t0 = time.time()
    pf1_c, pf1_o = nsga2_simple(
        tasks, vms, pop_size=70, n_gen=300,
        cx_rate=0.8, mut_rate=0.05,
        seed=42, verbose=True, ref=ref_algo)
    t1 = time.time()
    print(f"  Temps : {t1-t0:.1f}s  |  |Pareto| = {len(pf1_o)}")

    print("\n  [2/3] NSGA-II+HEFT Hybride (seed=42, n_gen=300)...")
    t2 = time.time()
    pf2_c, pf2_o = nsga2_heft_hybrid(
        tasks, vms, pop_size=70, n_gen=300,
        cx_rate=0.8, mut_rate=0.05,
        heft_ratio=0.3, ls_rate=0.4,
        seed=42, verbose=True, ref=ref_algo)
    t3 = time.time()
    print(f"  Temps : {t3-t2:.1f}s  |  |Pareto| = {len(pf2_o)}")

    # ── Métriques run principal ───────────────────────────────────────
    norm1_42 = normalize_to_unit(pf1_o, bounds)
    norm2_42 = normalize_to_unit(pf2_o, bounds)
    hv1_42   = compute_hv(norm1_42)
    hv2_42   = compute_hv(norm2_42)
    igd1_42  = compute_igd_plus(norm1_42, pstar)
    igd2_42  = compute_igd_plus(norm2_42, pstar)

    print("\n  [3/3] MOEA/D (seed=42, n_gen=300)...")
    t4 = time.time()
    pf3_c, pf3_o = moead(
        tasks, vms, H=8, T=5, n_gen=300,
        mut_rate=0.05, max_archive=70,
        seed=42, verbose=True)
    t5 = time.time()
    print(f"  Temps : {t5-t4:.1f}s  |  |Pareto| = {len(pf3_o)}")

    norm3_42 = normalize_to_unit(pf3_o, bounds)
    hv3_42   = compute_hv(norm3_42)
    igd3_42  = compute_igd_plus(norm3_42, pstar)

    W72 = 76
    print("\n" + "═"*W72)
    print("  MÉTRIQUES D'ÉVALUATION PARETO — 3 algorithmes")
    print("─"*W72)
    print(f"  Point de référence HV  : (1.1, 1.1, 1.1)")
    print(f"  Front P* (IGD+)        : {len(pstar)} solutions")
    print("─"*W72)
    print(f"  {'Métrique':<28} {'NSGA-II Simple':>16}  {'NSGA-II+HEFT':>14}  {'MOEA/D':>10}")
    print("─"*W72)
    def best3(v1,v2,v3,hi=True):
        best=max(v1,v2,v3) if hi else min(v1,v2,v3)
        return ["◄" if abs(v-best)<1e-12 else "  " for v in [v1,v2,v3]]
    b=best3(hv1_42,hv2_42,hv3_42,hi=True)
    print(f"  {'HV (↑ mieux)':<28} {hv1_42:>14.6f}{b[0]}  {hv2_42:>12.6f}{b[1]}  {hv3_42:>8.6f}{b[2]}")
    b=best3(igd1_42,igd2_42,igd3_42,hi=False)
    print(f"  {'IGD+ (↓ mieux)':<28} {igd1_42:>14.6f}{b[0]}  {igd2_42:>12.6f}{b[1]}  {igd3_42:>8.6f}{b[2]}")
    b=best3(len(pf1_o),len(pf2_o),len(pf3_o),hi=True)
    print(f"  {'|Pareto| (↑ mieux)':<28} {len(pf1_o):>16}{b[0]}  {len(pf2_o):>14}{b[1]}  {len(pf3_o):>10}{b[2]}")
    print("═"*W72)

    # ── Fronts détaillés ─────────────────────────────────────────────
    display_pareto("NSGA-II Simple   (seed=42)", pf1_c, pf1_o, tasks, vms)
    display_pareto("NSGA-II+HEFT Hybride (seed=42)", pf2_c, pf2_o, tasks, vms)
    display_pareto("MOEA/D (seed=42)", pf3_c, pf3_o, tasks, vms)

    # ── Graphiques ────────────────────────────────────────────────────
    print("\n  Génération des graphiques...")
    plot_pareto_fronts_3(pf1_o, pf2_o, pf3_o,
                          label1="NSGA-II Simple",
                          label2="NSGA-II+HEFT",
                          label3="MOEA/D",
                          save_path="pareto_workflow10.png")

    plot_metrics_bar_3(
        {
            "NSGA-II Simple": (avg(ga_hvs), avg(ga_igds), avg(ga_nf)),
            "NSGA-II+HEFT":   (avg(hb_hvs), avg(hb_igds), avg(hb_nf)),
            "MOEA/D":         (avg(md_hvs), avg(md_igds), avg(md_nf)),
        },
        save_path="metrics_bar_workflow10.png")

    plot_convergence_runs_3(ga_hvs, hb_hvs, md_hvs,
                             ga_igds, hb_igds, md_igds,
                             save_path="runs_workflow10.png")

    print("\n  ✓ Terminé. Fichiers générés :")
    print(f"    {OUT}/pareto_workflow10.png")
    print(f"    {OUT}/metrics_bar_workflow10.png")
    print(f"    {OUT}/runs_workflow10.png")
    print("═"*76)

╔══════════════════════════════════════════════════════════════════════╗
║  NSGA-II Simple  vs  NSGA-II+HEFT Hybride                           ║
║  Workflow 30 tâches × 5 VMs  |  20 runs  |  seed=42                   ║
╚══════════════════════════════════════════════════════════════════════╝

  Niveaux topologiques :
    Niveau 0 ( 5 tâches) : ['T1', 'T2', 'T3', 'T4']...
    Niveau 1 ( 9 tâches) : ['T10', 'T11', 'T12', 'T13']...
    Niveau 2 ( 5 tâches) : ['T15', 'T16', 'T17', 'T18']...
    Niveau 3 ( 4 tâches) : ['T20', 'T21', 'T22', 'T23']
    Niveau 4 ( 4 tâches) : ['T24', 'T25', 'T26', 'T27']
    Niveau 5 ( 2 tâches) : ['T28', 'T29']
    Niveau 6 ( 1 tâches) : ['T30']

  Calcul des bornes globales (500 éch., seed=999)...
  Makespan : [77.7394, 323.0000] s
  Coût     : [0.179444, 0.244697] $
  Énergie  : [8.613333, 10.277273] kWh

  Construction du front P* (2000 éch., seed=888)...
  P* : 179 solutions non-dominées (normalisé [0,1]³)

═════════════════════════════════════════════════